# Observatorio de Permanencia – ETL (Reto 1)
**Universidad Andina del Norte · Cohorte 2025-1**

Este notebook limpia e integra los cinco archivos originales de `Datos_Universidad_Andina.zip` y genera:

1. `cohorte_limpia.csv`: tabla limpia y anonimizada de la cohorte (1.200 estudiantes).
2. `registro_transformaciones.csv`: problema, regla y etapa de cada transformación.
3. `modelo/`: tablas del modelo estrella (1 tabla de hechos y 4 dimensiones) para el tablero.
4. `indicadores_clave.csv`: los 7 valores del tablero, para validar.

**Cómo ejecutarlo:** en Google Colab, menú *Entorno de ejecución → Ejecutar todas*. Si el zip no está cargado, la primera celda pide subirlo. El proceso es repetible: siempre parte de los CSV originales.

In [1]:
import os, re, zipfile, unicodedata
import pandas as pd
import numpy as np

ZIP = 'Datos_Universidad_Andina.zip'
DATOS = 'datos'
SALIDA = 'salida'

# En Google Colab, si el zip no existe, se pide subirlo
try:
    from google.colab import files
    if not os.path.exists(ZIP) and not os.path.isdir(DATOS):
        files.upload()
except ImportError:
    pass

if os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATOS)

os.makedirs(f'{SALIDA}/modelo', exist_ok=True)
registro = []  # registro de transformaciones

def regla(problema, regla_txt, etapa, archivo, filas):
    registro.append({'n': len(registro) + 1, 'problema': problema, 'regla': regla_txt,
                     'etapa': etapa, 'archivo': archivo, 'filas_afectadas': int(filas)})

## 1. Extracción

In [2]:
def leer(nombre):
    # utf-8-sig elimina el BOM; dtype=str evita que pandas altere documentos, códigos o notas
    return pd.read_csv(f'{DATOS}/{nombre}', dtype=str, encoding='utf-8-sig', keep_default_na=False)

sis = leer('sis_matriculas.csv')
adm = leer('admisiones.csv')
erp = leer('erp_pagos_2025_2.csv')
bie = leer('bienestar_registros_2025_1.csv')
moo = leer('moodle_actividad_2025_1.csv')

regla('Los archivos traen BOM y pandas puede convertir documentos y notas a número',
      'Leer con codificación utf-8-sig y todas las columnas como texto', 'Extracción', 'Todos',
      len(sis) + len(adm) + len(erp) + len(bie) + len(moo))
{n: len(d) for n, d in [('sis', sis), ('admisiones', adm), ('erp', erp), ('bienestar', bie), ('moodle', moo)]}

{'sis': 2282,
 'admisiones': 1480,
 'erp': 1132,
 'bienestar': 276,
 'moodle': 19200}

## 2. Transformación
### 2.1 SIS (matrículas)

In [3]:
MAPA_PROGRAMA = {
    'ADMON EMPRESAS': 'Administración de Empresas', 'PSICOLOGIA': 'Psicología',
    'ING. INFORMATICA': 'Ingeniería Informática', 'CONTADURIA': 'Contaduría Pública',
    'TRABAJO SOCIAL': 'Trabajo Social', 'ING INDUSTRIAL': 'Ingeniería Industrial',
    # abreviaturas de Bienestar
    'Adm.': 'Administración de Empresas', 'Psico.': 'Psicología', 'Ing. Inf.': 'Ingeniería Informática',
    'Cont.': 'Contaduría Pública', 'T. Social': 'Trabajo Social', 'Ing. Ind.': 'Ingeniería Industrial'}

# Documento con puntos de miles
n = sis['documento'].str.contains(r'\.').sum()
sis['documento'] = sis['documento'].str.replace('.', '', regex=False)
regla('Documento con puntos de miles (ej. 1.000.902.633)', 'Quitar los puntos del documento', 'Transformación', 'sis_matriculas', n)

# Periodo escrito en romano
n = (sis['periodo'] == '2025-II').sum()
sis['periodo'] = sis['periodo'].replace({'2025-II': '2025-2'})
regla('Periodo escrito "2025-II"', 'Normalizar "2025-II" a "2025-2"', 'Transformación', 'sis_matriculas', n)

# Programa con nombres distintos
n = sis['programa'].isin(MAPA_PROGRAMA).sum()
sis['programa'] = sis['programa'].replace(MAPA_PROGRAMA)
regla('Programa en mayúsculas o abreviado (ej. "ADMON EMPRESAS")', 'Llevar al nombre oficial con tabla de equivalencias', 'Transformación', 'sis_matriculas', n)

# Duplicados por código y periodo
n = sis.duplicated(['codigo', 'periodo']).sum()
sis = sis.drop_duplicates(['codigo', 'periodo'], keep='first')
regla('Filas duplicadas del mismo estudiante en el mismo periodo', 'Conservar una sola fila por código y periodo', 'Transformación', 'sis_matriculas', n)

# Promedio con coma decimal
n = sis['promedio_periodo'].str.contains(',').sum()
sis['promedio_periodo'] = sis['promedio_periodo'].str.replace(',', '.', regex=False)
regla('Promedio con coma decimal (ej. "2,6")', 'Reemplazar la coma por punto', 'Transformación', 'sis_matriculas', n)

# Promedio sin separador decimal (ej. "36" = 3,6)
sis['promedio_periodo'] = pd.to_numeric(sis['promedio_periodo'], errors='coerce')
n = (sis['promedio_periodo'] > 5).sum()
sis.loc[sis['promedio_periodo'] > 5, 'promedio_periodo'] /= 10
regla('Promedio sin separador decimal (ej. "36" en una escala de 0 a 5)', 'Dividir entre 10 los promedios mayores a 5', 'Transformación', 'sis_matriculas', n)

# Cohorte
cohorte = sis[(sis['periodo'] == '2025-1') & (sis['tipo_ingreso'] == 'Nuevo')].copy()
n = (sis['tipo_ingreso'] == 'Transferencia').sum()
regla('Transferencias que ingresan en 2025-2 mezcladas con la cohorte', 'Cohorte = periodo 2025-1 y tipo de ingreso "Nuevo"; se excluyen las transferencias', 'Transformación', 'sis_matriculas', n)

matriculados_2025_2 = set(sis.loc[sis['periodo'] == '2025-2', 'documento'])
cohorte['matriculado_2025_2'] = cohorte['documento'].isin(matriculados_2025_2).astype(int)
cohorte['desertor'] = 1 - cohorte['matriculado_2025_2']
cohorte = cohorte.sort_values('codigo').reset_index(drop=True)
print('Estudiantes en la cohorte:', len(cohorte))

Estudiantes en la cohorte: 1200


### 2.2 Admisiones

In [4]:
MESES = {'ene': '01', 'feb': '02', 'mar': '03', 'abr': '04', 'may': '05', 'jun': '06',
         'jul': '07', 'ago': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dic': '12'}

def fecha_iso(t):
    t = t.strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', t):                 # aaaa-mm-dd
        return t
    m = re.match(r'^(\d{2})/(\d{2})/(\d{4})$', t)             # dd/mm/aaaa
    if m:
        return f'{m[3]}-{m[2]}-{m[1]}'
    m = re.match(r'^(\d{2})-([a-zA-Z]{3})-(\d{4})$', t)        # dd-mes-aaaa
    if m:
        return f'{m[3]}-{MESES[m[2].lower()]}-{m[1]}'
    return None

def normalizar_nombre(t):
    # Mayúsculas, sin tildes y con espacios simples, para comparar nombres
    t = unicodedata.normalize('NFKD', t.upper().strip())
    t = ''.join(ch for ch in t if not unicodedata.combining(ch))
    return ' '.join(t.split())

# Duplicados: mismo documento con el nombre en mayúsculas
adm['es_mayus'] = adm['nombre_completo'] == adm['nombre_completo'].str.upper()
n = adm.duplicated('documento').sum()
adm = adm.sort_values('es_mayus').drop_duplicates('documento', keep='first').drop(columns='es_mayus')
regla('Admitidos duplicados con el nombre en mayúsculas', 'Conservar un registro por documento (el de nombre en formato título)', 'Transformación', 'admisiones', n)

# Fechas en tres formatos
n = (~adm['fecha_admision'].str.match(r'^\d{4}-\d{2}-\d{2}$')).sum()
adm['fecha_admision'] = adm['fecha_admision'].map(fecha_iso)
regla('Fechas de admisión en tres formatos (dd/mm/aaaa, dd-mes-aaaa, aaaa-mm-dd)', 'Convertir todas a aaaa-mm-dd', 'Transformación', 'admisiones', n)

adm['puntaje_saber11'] = pd.to_numeric(adm['puntaje_saber11'])
adm['estrato'] = adm['estrato'].astype(int)
adm['nombre_norm'] = adm['nombre_completo'].map(normalizar_nombre)
print('Admitidos únicos:', len(adm))

Admitidos únicos: 1450


### 2.3 ERP (pagos)

In [5]:
n = erp['documento'].str.contains(r'\.').sum()
erp['documento'] = erp['documento'].str.replace('.', '', regex=False)
regla('Documento con puntos de miles', 'Quitar los puntos del documento', 'Transformación', 'erp_pagos_2025_2', n)

n = (erp['estado_pago'] != erp['estado_pago'].str.title()).sum()
erp['estado_pago'] = erp['estado_pago'].str.title()
regla('Estado de pago sin estandarizar (PAGADO / Pagado)', 'Llevar el estado a formato título', 'Transformación', 'erp_pagos_2025_2', n)
erp['estado_pago'].value_counts()

estado_pago
Pagado       983
Pendiente    131
Anulado       18
Name: count, dtype: int64

### 2.4 Bienestar: cruce por nombre y programa, resolviendo homónimos

In [6]:
bie['programa'] = bie['programa'].replace(MAPA_PROGRAMA)
bie['nombre_norm'] = bie['nombre_completo'].map(normalizar_nombre)

# Homónimos: el mismo nombre con más de un documento en admisiones
homonimos = adm.groupby('nombre_norm')['documento'].nunique()
homonimos = homonimos[homonimos > 1].index
print('Nombres con homónimos en admisiones:', list(homonimos))

# Cruce por nombre Y programa: el programa distingue a los homónimos
bie = bie.merge(adm[['nombre_norm', 'programa', 'documento']], on=['nombre_norm', 'programa'], how='left')
sin_cruce = bie['documento'].isna().sum()
regla('Bienestar no trae documento y hay homónimos (mismo nombre en dos programas)',
      'Cruzar contra admisiones por nombre normalizado (mayúsculas, sin tildes) y programa', 'Transformación',
      'bienestar_registros_2025_1', len(bie) - sin_cruce)
print('Registros sin cruzar:', sin_cruce)

assert bie['documento'].notna().all() and len(bie) == len(bie.drop_duplicates(['nombre_completo', 'tipo_registro', 'fecha'])), 'Cruce ambiguo en Bienestar'
n = len(bie) - bie['documento'].nunique()
bie_est = (bie.dropna(subset=['documento'])
              .assign(v=1)
              .pivot_table(index='documento', columns='tipo_registro', values='v', aggfunc='max', fill_value=0))
regla('Bienestar tiene una fila por atención: un estudiante puede aparecer varias veces', 'Resumir en una fila por estudiante con indicadores 0/1 por tipo de registro', 'Transformación', 'bienestar_registros_2025_1', n)

Nombres con homónimos en admisiones: ['DIANA AGUDELO LOPEZ']
Registros sin cruzar: 0


### 2.5 Moodle: correos compartidos y escala de notas

In [7]:
# Moodle trae bloques de 16 semanas por estudiante, ordenados por código.
# Dos correos están compartidos por dos estudiantes cada uno: el correo no basta para identificarlos.
moo['bloque'] = moo.index // 16
moo['codigo'] = cohorte['codigo'].values[moo['bloque']]
coinciden = (moo['correo'].values == cohorte.set_index('codigo').loc[moo['codigo'], 'correo_institucional'].values).mean()
assert coinciden == 1.0, 'El orden de Moodle no coincide con el orden por código'
compartidos = cohorte['correo_institucional'].duplicated(keep=False).sum()
regla('Correos institucionales compartidos por dos estudiantes (ej. manuel.osorio94@)',
      'Asignar el código a cada bloque de 16 semanas según el orden por código; se valida que el correo coincida en el 100 % de las filas',
      'Transformación', 'moodle_actividad_2025_1', compartidos)

moo['semana'] = moo['semana'].astype(int)
moo['accesos'] = moo['accesos'].astype(int)
moo['nota'] = pd.to_numeric(moo['nota_parcial_0_100'], errors='coerce') / 20
regla('Nota parcial en escala de 0 a 100 y solo en las semanas 4, 8, 12 y 16', 'Convertir a escala de 0 a 5 (dividir entre 20)', 'Transformación', 'moodle_actividad_2025_1', moo['nota'].notna().sum())

acc = moo.pivot(index='codigo', columns='semana', values='accesos')
nota = moo.pivot(index='codigo', columns='semana', values='nota')
mood = pd.DataFrame({
    'accesos_s1_8': acc[list(range(1, 9))].sum(axis=1),
    'accesos_s5_8': acc[[5, 6, 7, 8]].sum(axis=1),
    'accesos_s7_8': acc[[7, 8]].sum(axis=1),
    'nota_s4': nota[4], 'nota_s8': nota[8]})
mood['nota_prom_s4_s8'] = (mood['nota_s4'] + mood['nota_s8']) / 2

### 2.6 Alerta temprana
- **Regla actual:** nota parcial promedio de las semanas 4 y 8 menor a 3,0, o 0 accesos sumando las semanas 7 y 8.
- **Regla nueva (Reto 5):** 7 accesos o menos en las semanas 5 a 8, o nota parcial promedio menor a 2,5 con 12 accesos o menos en las semanas 5 a 8.

In [8]:
cohorte = cohorte.merge(mood, left_on='codigo', right_index=True, how='left')
cohorte['alerta_actual'] = ((cohorte['nota_prom_s4_s8'] < 3.0) | (cohorte['accesos_s7_8'] == 0)).astype(int)
cohorte['alerta_nueva'] = ((cohorte['accesos_s5_8'] <= 7) |
                           ((cohorte['nota_prom_s4_s8'] < 2.5) & (cohorte['accesos_s5_8'] <= 12))).astype(int)
cohorte['acierto_actual'] = cohorte['alerta_actual'] * cohorte['desertor']
cohorte['acierto_nueva'] = cohorte['alerta_nueva'] * cohorte['desertor']

def matriz(col):
    return pd.crosstab(cohorte[col].map({1: 'Alertó', 0: 'No alertó'}),
                       cohorte['desertor'].map({1: 'Desertó', 0: 'No desertó'}))
print('Regla actual'); print(matriz('alerta_actual')); print()
print('Regla nueva'); print(matriz('alerta_nueva'))

Regla actual
desertor       Desertó  No desertó
alerta_actual                     
Alertó             121         250
No alertó           54         775

Regla nueva
desertor      Desertó  No desertó
alerta_nueva                     
Alertó             71          46
No alertó         104         979


### 2.7 Integración de la cohorte

In [9]:
FACULTAD = {'Administración de Empresas': 'Ciencias Económicas y Administrativas',
            'Contaduría Pública': 'Ciencias Económicas y Administrativas',
            'Ingeniería Informática': 'Ingeniería', 'Ingeniería Industrial': 'Ingeniería',
            'Psicología': 'Ciencias Sociales y Humanas', 'Trabajo Social': 'Ciencias Sociales y Humanas'}

cohorte = cohorte.merge(adm[['documento', 'estrato', 'tipo_colegio', 'puntaje_saber11', 'fecha_admision']],
                        on='documento', how='left')
cohorte['facultad'] = cohorte['programa'].map(FACULTAD)

cohorte['cancelacion_formal'] = cohorte['documento'].map(bie_est.get('Cancelación formal')).fillna(0).astype(int)
cohorte['alerta_docente'] = cohorte['documento'].map(bie_est.get('Alerta docente')).fillna(0).astype(int)

erp_valido = erp[erp['estado_pago'] != 'Anulado']
cohorte['pago_pendiente_2025_2'] = cohorte['documento'].isin(erp_valido.loc[erp_valido['estado_pago'] == 'Pendiente', 'documento']).astype(int)

assert len(cohorte) == 1200, 'La cohorte debe tener 1.200 estudiantes'
assert cohorte['estrato'].notna().all(), 'Todos los estudiantes de la cohorte deben cruzar con admisiones'
print('Cohorte integrada:', len(cohorte), 'estudiantes')

Cohorte integrada: 1200 estudiantes


## 3. Carga
### 3.1 Tabla limpia y anonimizada de la cohorte

In [10]:
# Clave sustituta para no publicar documento, código, correo ni nombre
adm = adm.sort_values('documento').reset_index(drop=True)
adm['sk_estudiante'] = ['EST' + str(i).zfill(4) for i in range(1, len(adm) + 1)]
cohorte = cohorte.merge(adm[['documento', 'sk_estudiante']], on='documento', how='left')

publica = cohorte[['sk_estudiante', 'programa', 'facultad', 'estrato', 'tipo_colegio', 'puntaje_saber11',
                   'creditos', 'promedio_periodo', 'matriculado_2025_2', 'desertor',
                   'accesos_s1_8', 'accesos_s5_8', 'accesos_s7_8', 'nota_s4', 'nota_s8', 'nota_prom_s4_s8',
                   'alerta_actual', 'alerta_nueva', 'cancelacion_formal', 'alerta_docente', 'pago_pendiente_2025_2']]
publica = publica.rename(columns={'promedio_periodo': 'promedio_2025_1'})
publica.to_csv(f'{SALIDA}/cohorte_limpia.csv', index=False, encoding='utf-8-sig')
regla('Datos personales y sensibles (documento, correo, nombre, atenciones psicológicas)',
      'Publicar la tabla con una clave sustituta, sin datos de identificación ni atenciones psicológicas', 'Carga',
      'cohorte_limpia.csv', len(publica))
print('cohorte_limpia.csv:', publica.shape)

cohorte_limpia.csv: (1200, 21)


### 3.2 Modelo estrella
**Granularidad:** una fila por estudiante admitido por periodo (2025-1 y 2025-2).
Se incluyen los 1.450 admitidos (no solo la cohorte) para poder calcular la absorción con los mismos filtros.

In [11]:
dim_periodo = pd.DataFrame({'periodo_id': [20251, 20252], 'periodo': ['2025-1', '2025-2'],
                            'anio': [2025, 2025], 'semestre': [1, 2]})

programas = sorted(FACULTAD)
dim_programa = pd.DataFrame({'programa_id': range(1, len(programas) + 1), 'programa': programas})
dim_programa['facultad'] = dim_programa['programa'].map(FACULTAD)

dim_estrato = pd.DataFrame({'estrato_id': range(1, 7), 'estrato': [f'Estrato {i}' for i in range(1, 7)],
                            'nivel_socioeconomico': ['Bajo', 'Bajo', 'Medio', 'Medio', 'Alto', 'Alto']})

dim_estudiante = adm[['sk_estudiante', 'tipo_colegio', 'puntaje_saber11', 'fecha_admision']].copy()

# Tabla de hechos
base = adm[['documento', 'sk_estudiante', 'programa', 'estrato']].merge(
    dim_programa[['programa', 'programa_id']], on='programa')
c = cohorte.set_index('documento')
filas = []
for periodo_id, periodo in [(20251, '2025-1'), (20252, '2025-2')]:
    f = base[['documento', 'sk_estudiante', 'programa_id', 'estrato']].rename(columns={'estrato': 'estrato_id'}).copy()
    f['periodo_id'] = periodo_id
    en_cohorte = f['documento'].isin(c.index)
    if periodo == '2025-1':
        f['es_admitido'] = 1
        f['matriculado'] = en_cohorte.astype(int)
        f['es_cohorte'] = en_cohorte.astype(int)
        for col in ['matriculado_2025_2', 'desertor', 'promedio_periodo', 'creditos', 'accesos_s5_8', 'nota_prom_s4_s8',
                    'alerta_actual', 'alerta_nueva', 'acierto_actual', 'acierto_nueva', 'cancelacion_formal', 'alerta_docente']:
            f[col] = f['documento'].map(c[col])
        f = f.rename(columns={'matriculado_2025_2': 'retenido'})
    else:
        f['es_admitido'] = 0
        f['matriculado'] = (en_cohorte & f['documento'].isin(matriculados_2025_2)).astype(int)
        f['es_cohorte'] = 0
        f['pago_pendiente'] = f['documento'].map(c['pago_pendiente_2025_2'])
    filas.append(f)
fact = pd.concat(filas, ignore_index=True).drop(columns='documento')
for col in ['retenido', 'desertor', 'alerta_actual', 'alerta_nueva', 'acierto_actual', 'acierto_nueva',
            'cancelacion_formal', 'alerta_docente', 'pago_pendiente']:
    fact[col] = fact[col].fillna(0).astype(int)

for nombre, df in [('fact_matricula', fact), ('dim_estudiante', dim_estudiante), ('dim_programa', dim_programa),
                   ('dim_periodo', dim_periodo), ('dim_estrato', dim_estrato)]:
    df.to_csv(f'{SALIDA}/modelo/{nombre}.csv', index=False, encoding='utf-8-sig')
print('fact_matricula:', fact.shape)

fact_matricula: (2900, 20)


### 3.3 Indicadores clave y registro de transformaciones

In [12]:
def tasa(num, den):
    return round(100 * num / den, 2)

f1 = fact[fact['periodo_id'] == 20251]
ind = [
    ('Absorción (%)', tasa(f1['es_cohorte'].sum(), f1['es_admitido'].sum())),
    ('Retención de primer periodo (%)', tasa(f1['retenido'].sum(), f1['es_cohorte'].sum())),
    ('Deserción (%)', tasa(f1['desertor'].sum(), f1['es_cohorte'].sum())),
    ('Promedio de la cohorte (0 a 5)', round(cohorte['promedio_periodo'].mean(), 2)),
    ('Alertados - regla actual', int(cohorte['alerta_actual'].sum())),
    ('Precisión de la alerta - regla actual (%)', tasa(cohorte['acierto_actual'].sum(), cohorte['alerta_actual'].sum())),
    ('Cobertura de la alerta - regla actual (%)', tasa(cohorte['acierto_actual'].sum(), cohorte['desertor'].sum())),
    ('Alertados - regla nueva', int(cohorte['alerta_nueva'].sum())),
    ('Precisión de la alerta - regla nueva (%)', tasa(cohorte['acierto_nueva'].sum(), cohorte['alerta_nueva'].sum())),
    ('Cobertura de la alerta - regla nueva (%)', tasa(cohorte['acierto_nueva'].sum(), cohorte['desertor'].sum())),
]
for prog, g in cohorte.groupby('programa'):
    ind.append((f'Deserción {prog} (%)', tasa(g['desertor'].sum(), len(g))))
indicadores = pd.DataFrame(ind, columns=['indicador', 'valor'])
indicadores.to_csv(f'{SALIDA}/indicadores_clave.csv', index=False, encoding='utf-8-sig')

reg = pd.DataFrame(registro)
reg.to_csv(f'{SALIDA}/registro_transformaciones.csv', index=False, encoding='utf-8-sig')
print(indicadores.to_string(index=False))
print()
print(reg[['n', 'problema', 'regla', 'etapa', 'filas_afectadas']].to_string(index=False))

                                indicador  valor
                            Absorción (%)  82.76
          Retención de primer periodo (%)  85.42
                            Deserción (%)  14.58
           Promedio de la cohorte (0 a 5)   3.38
                 Alertados - regla actual 371.00
Precisión de la alerta - regla actual (%)  32.61
Cobertura de la alerta - regla actual (%)  69.14
                  Alertados - regla nueva 117.00
 Precisión de la alerta - regla nueva (%)  60.68
 Cobertura de la alerta - regla nueva (%)  40.57
 Deserción Administración de Empresas (%)  18.11
         Deserción Contaduría Pública (%)  14.29
      Deserción Ingeniería Industrial (%)  12.50
     Deserción Ingeniería Informática (%)  16.53
                 Deserción Psicología (%)  10.53
             Deserción Trabajo Social (%)  14.47

 n                                                                          problema                                                                                  

**Descarga (en Colab):** la siguiente celda comprime la carpeta `salida` para descargarla.

In [13]:
import shutil
shutil.make_archive('salida_observatorio', 'zip', SALIDA)
try:
    from google.colab import files
    files.download('salida_observatorio.zip')
except ImportError:
    print('Archivo generado: salida_observatorio.zip')

Archivo generado: salida_observatorio.zip
